# Noise Gate: catching poisoned images **before** Stable Diffusion training

You already built a tiled, batched, gradient-free noise detector in
[`../04-Adversarial-Attacks/tiled_noise_attack_detection.ipynb`](../04-Adversarial-Attacks/tiled_noise_attack_detection.ipynb):
cut an image into tiles, score each tile's high-frequency energy, flag the tiles that carry
injected FGSM/PGD noise. This notebook puts that detector to work in the **Stable Diffusion**
setting.

**The problem.** Stable Diffusion is trained on huge scraped image sets. An attacker can slip in
**poisoned images** (adversarial / backdoor / Nightshade-Glaze-style perturbations). Training on
them corrupts the model. The SD training path is:

```
image  ->  VAE.encode  ->  add diffusion noise (scheduler)  ->  UNet learns to denoise
```

**The fix (this notebook).** Put a **gate right at the front** of that path. Every incoming image
is tiled and scanned; images whose tiles look tampered are **rejected before they ever reach the
VAE / training loop**. The flagged tiles also tell you *where* the poison is.

**Why gradient-free & batched (the paper).** Following
**ViT-ReciproCAM** ([arXiv:2310.02588](https://arxiv.org/abs/2310.02588)) — saliency *without*
gradients or attention — the gate uses a **model-free, forward-only** score, so a whole batch of
images clears in one pass. Cheap enough to run on every training image.

> Kaggle/Colab: set accelerator to **GPU T4 x2** and turn **Internet On**. Falls back to a few web
> sample images if no dataset is mounted.

## 0. Install & imports
`grad-cam` is only needed if you later swap in a CAM backbone; the gate itself needs just
torch + torchvision. `diffusers` is optional (used only for the real SD VAE demo at the end).

In [ ]:
!pip install -q torch torchvision diffusers  # diffusers optional (SD VAE demo only)

In [ ]:
# ============================================================
# CELL 1 - Imports & GPU inventory
# ============================================================
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as patches
from torchvision import models, transforms
from PIL import Image, ImageDraw
import urllib.request, os
from concurrent.futures import ThreadPoolExecutor

N_GPU = torch.cuda.device_count()
DEVICES = [f"cuda:{i}" for i in range(N_GPU)] if N_GPU else ["cpu"]
print("GPUs found:", N_GPU)
for i in range(N_GPU):
    print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")
print("Using devices:", DEVICES)

## 1. The reusable tiler + helpers
Straight from your attack notebook. Images live in `[0,1]` pixel space so any injected noise stays
measured in real pixels. `tile_image` cuts `[1,3,SIZE,SIZE]` into a `GRID x GRID` stack of tiles.

In [ ]:
# ============================================================
# CELL 2 - Preprocessing, tiler, helpers (reused from the attack notebook)
# ============================================================
SIZE = 448            # working resolution per image
GRID = 4              # GRID x GRID tiles -> 16 tiles/image
TILE = SIZE // GRID   # 112 px per tile

to_tensor = transforms.Compose([transforms.Resize((SIZE, SIZE)), transforms.ToTensor()])

def load_image(path_or_url):
    # local path OR url -> [1,3,SIZE,SIZE] float tensor in [0,1] on CPU
    if str(path_or_url).startswith("http"):
        fn = "/tmp/" + os.path.basename(path_or_url)
        if not os.path.exists(fn):
            urllib.request.urlretrieve(path_or_url, fn)
        path_or_url = fn
    return to_tensor(Image.open(path_or_url).convert("RGB")).unsqueeze(0)

def tile_image(x, grid=GRID, tile=TILE):
    # [1,3,SIZE,SIZE] -> [grid*grid, 3, tile, tile]  (row-major)
    p = x.unfold(2, tile, tile).unfold(3, tile, tile)
    p = p.permute(0, 2, 3, 1, 4, 5).reshape(-1, 3, tile, tile)
    return p.contiguous()

def tile_grid_to_full(per_tile, grid=GRID, tile=TILE):
    g = np.asarray(per_tile).reshape(grid, grid)
    return np.kron(g, np.ones((tile, tile)))

def to_np(t):   return t.squeeze().detach().cpu().permute(1, 2, 0).numpy()
def norm01(a):
    a = np.asarray(a, np.float32); return (a - a.min()) / (a.max() - a.min() + 1e-8)

print(f"tiler ready: {GRID}x{GRID} grid, {TILE}x{TILE}px tiles, {GRID*GRID} tiles/image")

## 2. The detector + the **gate**
Same high-frequency-energy score as your notebook: FGSM/PGD noise is high-frequency, so we score
each tile by `mean(|tile - blur(tile)|)`.

Two functions build on it:
- `clean_threshold(...)` — calibrate a threshold **once on trusted clean tiles** (`median + k*MAD`).
- `noise_gate(images)` — the deployable gate: tile every image, flag tiles over the threshold, and
  **reject an image** when more than `REJECT_FRAC` of its tiles are flagged. Returns, per image, a
  keep/reject verdict *and* the flagged-tile map (localization).

In [ ]:
# ============================================================
# CELL 3 - HF-energy per-tile score, clean calibration, and the GATE
# ============================================================
def _gaussian_kernel(sigma=1.0, ksize=5):
    ax = torch.arange(ksize) - ksize // 2
    g = torch.exp(-(ax**2) / (2*sigma**2)); g = g / g.sum()
    return torch.outer(g, g).view(1, 1, ksize, ksize).repeat(3, 1, 1, 1)  # depthwise, 3ch
_GK = _gaussian_kernel()

def hf_energy_per_tile(tiles):
    # tiles:[B,3,h,w] in [0,1] -> [B] high-frequency energy (one score per tile)
    k = _GK.to(tiles.device)
    blur = F.conv2d(tiles, k, padding=k.shape[-1]//2, groups=3)
    return (tiles - blur).abs().mean(dim=(1, 2, 3)).cpu().numpy()

def clean_threshold(clean_images, k=3.0):
    # calibrate ONCE on trusted clean images: robust median + k*MAD baseline
    scores = np.concatenate([hf_energy_per_tile(tile_image(x)) for x in clean_images])
    med = np.median(scores); mad = np.median(np.abs(scores - med)) + 1e-8
    return med + k * 1.4826 * mad

REJECT_FRAC = 0.10   # reject an image if > this fraction of its tiles are flagged

def noise_gate(images, thresh, reject_frac=REJECT_FRAC):
    # scan each image tile-by-tile against the fixed clean threshold.
    # returns list of dicts: keep verdict + per-tile score + per-tile flag map.
    out = []
    for x in images:
        hf = hf_energy_per_tile(tile_image(x))
        flagged = hf > thresh
        frac = flagged.mean()
        out.append(dict(scores=hf, flagged=flagged,
                        frac_flagged=float(frac), keep=bool(frac <= reject_frac)))
    return out

## 3. Poison generator (reused attack)
To test the gate we need poisoned images. Reusing your approach: confine an FGSM/PGD perturbation
to an **irregular blob** (not the whole image, not tile-aligned) so the gate has to find *which
tiles* it touched. A small ResNet supplies the gradient for the attack; the **gate itself uses no
model**.

In [ ]:
# ============================================================
# CELL 4 - Small model + irregular-region FGSM/PGD poison (reused from attack notebook)
# ============================================================
_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
def normalize(t): return (t - _mean.to(t.device)) / _std.to(t.device)

_D = DEVICES[0]
_ATK_MODEL = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1).eval().to(_D)

def random_blob_mask(size=SIZE, seed=None, n_verts=9, rad_frac=(0.12, 0.34)):
    rng = np.random.default_rng(seed)
    cx, cy = rng.uniform(0.30, 0.70, 2) * size
    angles = np.sort(rng.uniform(0, 2*np.pi, n_verts))
    radii = rng.uniform(*rad_frac, n_verts) * size
    pts = [(float(cx + r*np.cos(a)), float(cy + r*np.sin(a))) for a, r in zip(angles, radii)]
    im = Image.new("L", (size, size), 0); ImageDraw.Draw(im).polygon(pts, fill=1)
    return torch.tensor(np.array(im), dtype=torch.float32).view(1, 1, size, size)

def _grad_sign(xi):
    out = _ATK_MODEL(normalize(xi))
    loss = F.cross_entropy(out, out.argmax(1).detach())   # push away from current prediction
    _ATK_MODEL.zero_grad(); loss.backward()
    return xi.grad.sign()

def poison(x, method="pgd", epsilon=0.05, alpha=0.01, steps=10, mask=None):
    x0 = x.clone().detach().to(_D); xi = x0.clone()
    m = mask.to(_D) if mask is not None else None
    if method == "fgsm":
        xi.requires_grad_(True)
        step = epsilon * _grad_sign(xi)
        if m is not None: step = step * m
        return torch.clamp(xi.detach() + step, 0, 1).cpu()
    for _ in range(steps):                                  # PGD
        xi.requires_grad_(True)
        step = alpha * _grad_sign(xi)
        if m is not None: step = step * m
        with torch.no_grad():
            xi = torch.min(torch.max(xi.detach() + step, x0 - epsilon), x0 + epsilon)
            xi = torch.clamp(xi, 0, 1)
    return xi.detach().cpu()

def mask_tile_coverage(mask, cover=0.05):
    mt = tile_image(mask.repeat(1, 3, 1, 1))
    return mt.mean(dim=(1, 2, 3)).numpy() > cover           # ground-truth attacked tiles

## 4. Build a mixed training batch (some clean, some poisoned)
This simulates a scraped training set where some images have been tampered with. We calibrate the
gate on a **held-out clean set**, then feed it the mixed batch.

In [ ]:
# ============================================================
# CELL 5 - Gather images; poison a subset -> a realistic dirty training batch
# ============================================================
base = "https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/"
NAMES = ["n02099601_golden_retriever.JPEG", "n02123045_tabby.JPEG", "n02391049_zebra.JPEG",
         "n02129165_lion.JPEG", "n02129604_tiger.JPEG", "n02510455_giant_panda.JPEG",
         "n01518878_ostrich.JPEG", "n01806143_peacock.JPEG", "n01882714_koala.JPEG",
         "n02007558_flamingo.JPEG"]
raw = [load_image(base + n) for n in NAMES]

# 4 trusted-clean images to calibrate the gate on (never poisoned)
CLEAN_CALIB = raw[:4]
K_SENSITIVITY = 3.0
THRESH = clean_threshold(CLEAN_CALIB, k=K_SENSITIVITY)
print(f"Gate calibrated on {len(CLEAN_CALIB)} clean images -> threshold = {THRESH:.5f}")

# incoming training batch = remaining 6 images; poison every other one
ATTACK_METHOD, EPSILON = "pgd", 0.05
incoming, truth_masks, is_poisoned = [], [], []
for i, x in enumerate(raw[4:]):
    if i % 2 == 0:                       # poison half of them
        mask = random_blob_mask(seed=i)
        incoming.append(poison(x, method=ATTACK_METHOD, epsilon=EPSILON, mask=mask))
        truth_masks.append(mask); is_poisoned.append(True)
    else:
        incoming.append(x); truth_masks.append(None); is_poisoned.append(False)
print(f"Incoming batch: {len(incoming)} images, {sum(is_poisoned)} actually poisoned")

## 5. Run the gate & visualize what it blocks
For each incoming image: original, the per-tile noise score, and the gate's decision (flagged tiles
in cyan; whole-image **REJECTED / kept** verdict in the title). Poisoned images should light up and
get rejected *before* any of them reaches the VAE.

In [ ]:
# ============================================================
# CELL 6 - Gate the incoming batch + per-image figure
# ============================================================
verdicts = noise_gate(incoming, THRESH)

def _outline(ax, ids, color, ls="-"):
    for kk in np.where(ids)[0]:
        r, c = divmod(int(kk), GRID)
        ax.add_patch(patches.Rectangle((c*TILE, r*TILE), TILE, TILE,
                     fill=False, edgecolor=color, linewidth=2.5, linestyle=ls))

for i, (x, v) in enumerate(zip(incoming, verdicts)):
    fig, ax = plt.subplots(1, 3, figsize=(13, 4.4))
    ax[0].imshow(to_np(x))
    if truth_masks[i] is not None:
        ax[0].contour(truth_masks[i].squeeze().numpy(), levels=[0.5], colors="yellow", linewidths=2)
    ax[0].set_title("Incoming image\n(yellow = real poison region)")
    ax[1].imshow(tile_grid_to_full(norm01(v["scores"])), cmap="hot")
    ax[1].set_title("Per-tile noise score")
    ax[2].imshow(to_np(x)); _outline(ax[2], v["flagged"], "cyan", ls="--")
    ax[2].set_title("Gate decision\ncyan-- = flagged tiles")
    for a in ax: a.axis("off")
    truth = "POISONED" if is_poisoned[i] else "clean"
    verd  = "kept -> train" if v["keep"] else "REJECTED (blocked)"
    fig.suptitle(f"truth: {truth}   |   gate: {verd}   "
                 f"({v['frac_flagged']:.0%} of tiles flagged)", y=1.02, fontsize=12)
    plt.tight_layout(); plt.show()

## 6. Gate scorecard
Did the gate keep the clean images and reject the poisoned ones? This is the whole point: a clean
train/reject decision made **before** training.

In [ ]:
# ============================================================
# CELL 7 - Scorecard: gate verdict vs ground truth
# ============================================================
import pandas as pd
rows = []
for i, v in enumerate(verdicts):
    rows.append(dict(image=NAMES[4+i][:24], truth="poisoned" if is_poisoned[i] else "clean",
                     frac_flagged=round(v["frac_flagged"], 3),
                     verdict="KEEP" if v["keep"] else "REJECT",
                     correct=(v["keep"] != is_poisoned[i])))
df = pd.DataFrame(rows)
acc = df["correct"].mean()
caught = sum((not verdicts[i]["keep"]) and is_poisoned[i] for i in range(len(verdicts)))
false_block = sum((not verdicts[i]["keep"]) and (not is_poisoned[i]) for i in range(len(verdicts)))
print(f"Gate accuracy: {acc:.0%}   |   poisoned caught: {caught}/{sum(is_poisoned)}   "
      f"|   clean wrongly blocked: {false_block}/{sum(~np.array(is_poisoned))}")
df

## 7. Wire the gate into the real Stable Diffusion path (optional)
This mirrors the custom pipeline from the SD intro notebook: `VAE.encode -> scheduler.add_noise`.
The gate runs **first**; only images it keeps get encoded to latents and noised for training.
Guarded with try/except so the notebook still runs if the diffusers weights aren't available.

In [ ]:
# ============================================================
# CELL 8 - gate -> VAE.encode -> add diffusion noise  (only KEPT images proceed)
# ============================================================
try:
    from diffusers import AutoencoderKL, DDPMScheduler
    vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae").to(_D).eval()
    sched = DDPMScheduler.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="scheduler")

    kept = [x for x, v in zip(incoming, verdicts) if v["keep"]]
    blocked = len(incoming) - len(kept)
    print(f"Gate passed {len(kept)} images to training, blocked {blocked}.")

    with torch.no_grad():
        batch = torch.cat([F.interpolate(x, size=(512, 512)) for x in kept]).to(_D)
        latents = vae.encode(batch * 2 - 1).latent_dist.sample() * 0.18215   # [B,4,64,64]
        noise = torch.randn_like(latents)
        t = torch.randint(0, sched.config.num_train_timesteps, (latents.shape[0],), device=_D)
        noisy_latents = sched.add_noise(latents, noise, t)                    # forward diffusion
    print(f"Encoded to latents {tuple(latents.shape)} and added diffusion noise -> "
          f"{tuple(noisy_latents.shape)}. These clean latents are what the UNet would train on.")
except Exception as e:
    print("SD VAE demo skipped (weights unavailable or OOM):", repr(e))
    print("The gate logic above is independent of this cell and still holds.")

## Recap & knobs

**What this adds to the Stable Diffusion notebook:** a **noise gate** in front of the training
pipeline, built from your Grad-CAM attack work — tile every incoming image, score high-frequency
energy per tile (gradient-free, batched, following ViT-ReciproCAM), and **reject poisoned images
before `VAE.encode`** so the diffusion model never trains on them.

**Knobs**
- `GRID` — tile granularity (finer localizes better, costs more tiles).
- `K_SENSITIVITY` — threshold `median + k*MAD`; lower = more sensitive gate.
- `REJECT_FRAC` — how many flagged tiles reject a whole image.
- `ATTACK_METHOD` / `EPSILON` — poison strength in the demo (`fgsm` fast, `pgd` stronger/subtler).

**Deployment note:** calibrate `THRESH` **once** on a trusted clean set (done here on `CLEAN_CALIB`),
then apply that fixed threshold to the live training stream.

**Next steps**
- Replace HF-energy with a small learned per-tile classifier for low-frequency / subtler poisons.
- Swap in the paper's real ViT + ReciproCAM as the gradient-free saliency backbone.
- Move `noise_gate` into a `DataLoader` collate function so it filters a real training stream.